# Pipeline: Counseling Session Effectiveness

## Problem framing
**Who cares**: Social workers / case management lead.

**Business question**: Which session types, durations, and approaches produce the best emotional outcomes? Which residents need a different approach?

**Predictive goal**: Predict `emotional_improvement` — the change in emotional state from session start to end (regression on ordinal emotional scale).

## Emotional scale
Distressed (1) → Angry (2) → Sad (3) → Anxious (4) → Withdrawn (5) → Calm (6) → Hopeful (7) → Happy (8)

## Data used
- `process_recordings.csv` — 2,819 sessions with start/end emotional states, session type, duration, social worker
- `residents.csv` — full resident profile including abuse subcategories, family context
- `incident_reports.csv` — recent incident context
- `home_visitations.csv` — recent visit context

## Output
- `predictions_counseling_effectiveness.csv` — per-session predictions + best practices

In [1]:
from __future__ import annotations
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.metrics import mean_absolute_error, r2_score, roc_auc_score, classification_report
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", 200)

HERE = Path.cwd()
if (HERE / "process_recordings.csv").exists():
    DATA_DIR = HERE
elif (HERE.parent / "process_recordings.csv").exists():
    DATA_DIR = HERE.parent
else:
    raise FileNotFoundError("Could not locate process_recordings.csv")

sessions = pd.read_csv(DATA_DIR / "process_recordings.csv")
residents = pd.read_csv(DATA_DIR / "residents.csv")
incidents = pd.read_csv(DATA_DIR / "incident_reports.csv")
visits = pd.read_csv(DATA_DIR / "home_visitations.csv")
safehouses = pd.read_csv(DATA_DIR / "safehouses.csv")

sessions["session_date"] = pd.to_datetime(sessions["session_date"], errors="coerce")
incidents["incident_date"] = pd.to_datetime(incidents["incident_date"], errors="coerce")
visits["visit_date"] = pd.to_datetime(visits["visit_date"], errors="coerce")

print(f"Sessions: {len(sessions)}, Residents: {len(residents)}")
print(f"Emotional states observed:\n{sessions['emotional_state_observed'].value_counts().to_string()}")

Sessions: 2819, Residents: 60
Emotional states observed:
emotional_state_observed
Sad           499
Calm          476
Anxious       462
Angry         392
Hopeful       391
Withdrawn     356
Happy         150
Distressed     93


In [2]:
# Build features per session

EMO_RANK = {
    "Distressed": 1, "Angry": 2, "Sad": 3, "Anxious": 4,
    "Withdrawn": 5, "Calm": 6, "Hopeful": 7, "Happy": 8,
}

df = sessions.copy()
df["emo_start"] = df["emotional_state_observed"].map(EMO_RANK)
df["emo_end"] = df["emotional_state_end"].map(EMO_RANK)
df["emo_improvement"] = df["emo_end"] - df["emo_start"]

# Drop rows with unmapped emotions
df = df.dropna(subset=["emo_start", "emo_end"]).copy()

# Session-level features
df["session_duration_minutes"] = pd.to_numeric(df["session_duration_minutes"], errors="coerce")
df["concerns_flagged"] = df["concerns_flagged"].fillna(False).astype(bool).astype(int)
df["referral_made"] = df["referral_made"].fillna(False).astype(bool).astype(int)
df["progress_noted"] = df["progress_noted"].fillna(False).astype(bool).astype(int)

# Recent context: incidents and visits in the 30 days before this session
LOOKBACK = 30

def recent_context(row):
    rid = row["resident_id"]
    sdate = row["session_date"]
    window_start = sdate - pd.Timedelta(days=LOOKBACK)
    
    inc = incidents[(incidents["resident_id"] == rid) & 
                    (incidents["incident_date"] >= window_start) & 
                    (incidents["incident_date"] < sdate)]
    vis = visits[(visits["resident_id"] == rid) & 
                 (visits["visit_date"] >= window_start) & 
                 (visits["visit_date"] < sdate)]
    prior_sess = sessions[(sessions["resident_id"] == rid) & 
                          (sessions["session_date"] >= window_start) & 
                          (sessions["session_date"] < sdate)]
    
    return pd.Series({
        "incidents_last_30d": len(inc),
        "visits_last_30d": len(vis),
        "prior_sessions_30d": len(prior_sess),
    })

context = df.apply(recent_context, axis=1)
df = pd.concat([df, context], axis=1)

# Resident static features (full profile)
res_cols = [
    "resident_id", "safehouse_id", "case_status", "case_category",
    "initial_risk_level", "current_risk_level", "referral_source",
    "has_special_needs", "is_pwd", "sex",
    "sub_cat_orphaned", "sub_cat_trafficked", "sub_cat_child_labor",
    "sub_cat_physical_abuse", "sub_cat_sexual_abuse", "sub_cat_osaec",
    "sub_cat_cicl", "sub_cat_at_risk", "sub_cat_street_child",
    "family_is_4ps", "family_solo_parent", "family_indigenous",
    "family_informal_settler", "age_upon_admission", "present_age",
]
res_cols = [c for c in res_cols if c in residents.columns]
df = df.merge(residents[res_cols], on="resident_id", how="left")

print(f"Model dataset: {len(df)} rows")
print(f"Avg emotional improvement: {df['emo_improvement'].mean():.2f} (scale: -7 to +7)")
print(f"Positive improvement rate: {(df['emo_improvement'] > 0).mean():.1%}")

Model dataset: 2819 rows
Avg emotional improvement: 1.92 (scale: -7 to +7)
Positive improvement rate: 77.4%


In [3]:
# Model: predict emotional improvement (regression)

TARGET = "emo_improvement"
exclude = {
    "recording_id", "resident_id", "session_date", "social_worker",
    "session_narrative", "interventions_applied", "follow_up_actions",
    "notes_restricted", "emotional_state_observed", "emotional_state_end",
    "emo_end", TARGET, "safehouse_id",
}
feature_cols = [c for c in df.columns if c not in exclude]

X_all = df[feature_cols].copy()
y_all = df[TARGET].astype(float)

X_train, X_test, y_train, y_test = train_test_split(X_all, y_all, test_size=0.2, random_state=42)

numeric_cols = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
cat_cols = [c for c in X_train.columns if c not in numeric_cols]

for c in cat_cols:
    X_train[c] = X_train[c].astype("object")
    X_test[c] = X_test[c].astype("object")

pre = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), numeric_cols),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("ohe", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
])

models = {
    "Ridge": Ridge(alpha=1.0, random_state=42),
    "RandomForest": RandomForestRegressor(n_estimators=300, max_depth=12, random_state=42, n_jobs=-1),
}

best_name, best_score, best_pipe = None, -1e9, None
for name, model in models.items():
    p = Pipeline([("pre", pre), ("model", model)])
    cv = cross_val_score(p, X_train, y_train, cv=5, scoring="r2")
    print(f"{name}: CV R² = {cv.mean():.3f} ± {cv.std():.3f}")
    if cv.mean() > best_score:
        best_name, best_score, best_pipe = name, cv.mean(), p

print(f"\nBest model: {best_name}")
best_pipe.fit(X_train, y_train)
pipe = best_pipe

pred = pipe.predict(X_test)
print(f"Test MAE: {mean_absolute_error(y_test, pred):.3f} points")
print(f"Test R²:  {r2_score(y_test, pred):.3f}")
print(f"\nTarget stats — mean: {y_all.mean():.2f}, std: {y_all.std():.2f}")

Ridge: CV R² = 0.591 ± 0.018


RandomForest: CV R² = 0.695 ± 0.012

Best model: RandomForest


Test MAE: 0.793 points
Test R²:  0.667

Target stats — mean: 1.92, std: 1.76


In [4]:
# Feature importance

ohe = pipe.named_steps["pre"].named_transformers_["cat"].named_steps["ohe"]
cat_feature_names = ohe.get_feature_names_out(cat_cols).tolist() if len(cat_cols) else []
feat_names = numeric_cols + cat_feature_names

model = pipe.named_steps["model"]
if hasattr(model, "coef_"):
    imp = model.coef_
    label = "coef"
else:
    imp = model.feature_importances_
    label = "importance"

imp_df = pd.DataFrame({"feature": feat_names, label: imp}).sort_values(label, key=abs, ascending=False)
print(f"Top 25 features by |{label}| ({best_name})")
display(imp_df.head(25))

Top 25 features by |importance| (RandomForest)


,feature,importance
4,emo_start,0.801357
0,session_duration_minutes,0.046463
7,prior_sessions_30d,0.019723
6,visits_last_30d,0.012948
2,concerns_flagged,0.005352
3,referral_made,0.004670
5,incidents_last_30d,0.003540
29,case_category_Foundling,0.003034
11,sub_cat_trafficked,0.002764
17,sub_cat_at_risk,0.002474


In [5]:
# Best practices analysis: which session types work best for which emotional states?

best_practices = (
    df.groupby(["emotional_state_observed", "session_type"])
    .agg(
        avg_improvement=("emo_improvement", "mean"),
        session_count=("emo_improvement", "count"),
        avg_duration=("session_duration_minutes", "mean"),
    )
    .reset_index()
    .sort_values(["emotional_state_observed", "avg_improvement"], ascending=[True, False])
)

print("=== BEST SESSION TYPES BY STARTING EMOTIONAL STATE ===")
for state in EMO_RANK:
    sub = best_practices[best_practices["emotional_state_observed"] == state]
    if sub.empty:
        continue
    best = sub.iloc[0]
    print(f"\n  {state} → Best: {best['session_type']} "
          f"(avg improvement: +{best['avg_improvement']:.1f}, "
          f"n={int(best['session_count'])}, "
          f"avg {best['avg_duration']:.0f} min)")

# Output predictions for the most recent session per resident
df = df.merge(safehouses[["safehouse_id", "safehouse_code", "name"]], on="safehouse_id", how="left")

latest = df.sort_values(["resident_id", "session_date"]).groupby("resident_id").tail(1).copy()

X_latest = latest[feature_cols].copy()
for c in cat_cols:
    if c in X_latest.columns:
        X_latest[c] = X_latest[c].astype("object")

latest["pred_emo_improvement"] = pipe.predict(X_latest)

out_cols = [
    "resident_id", "safehouse_code", "name", "session_date",
    "emotional_state_observed", "emo_start", "session_type",
    "session_duration_minutes", "pred_emo_improvement",
    "incidents_last_30d", "prior_sessions_30d",
    "current_risk_level", "case_category",
]
out_cols = [c for c in out_cols if c in latest.columns]

out = latest[out_cols].sort_values("pred_emo_improvement").copy()

out_path = DATA_DIR / "predictions_counseling_effectiveness.csv"
out.to_csv(out_path, index=False)

print(f"\nSaved: {out_path}")
print(f"Residents with predicted low improvement (< 1.0): {(out['pred_emo_improvement'] < 1.0).sum()}")
display(out.head(15))

=== BEST SESSION TYPES BY STARTING EMOTIONAL STATE ===

  Distressed → Best: Individual (avg improvement: +3.8, n=56, avg 62 min)

  Angry → Best: Individual (avg improvement: +3.9, n=242, avg 60 min)

  Sad → Best: Group (avg improvement: +3.5, n=172, avg 83 min)

  Anxious → Best: Individual (avg improvement: +2.5, n=303, avg 60 min)

  Withdrawn → Best: Group (avg improvement: +0.2, n=144, avg 85 min)

  Calm → Best: Group (avg improvement: +1.1, n=155, avg 83 min)

  Hopeful → Best: Individual (avg improvement: +0.5, n=250, avg 59 min)

  Happy → Best: Individual (avg improvement: +-0.5, n=94, avg 62 min)

Saved: /Users/masonzarges/Desktop/INTEX Data/predictions_counseling_effectiveness.csv
Residents with predicted low improvement (< 1.0): 29


,resident_id,safehouse_code,name,session_date,emotional_state_observed,emo_start,session_type,session_duration_minutes,pred_emo_improvement,incidents_last_30d,prior_sessions_30d,current_risk_level,case_category
363,6,SH05,Lighthouse Safehouse 5,2026-02-27,Withdrawn,5,Individual,86,-1.015390,0,3,Medium,Foundling
587,10,SH01,Lighthouse Safehouse 1,2026-02-21,Happy,8,Group,71,-0.780999,0,2,Low,Surrendered
508,9,SH07,Lighthouse Safehouse 7,2026-02-21,Happy,8,Group,97,-0.690044,0,1,Low,Surrendered
2562,54,SH07,Lighthouse Safehouse 7,2024-06-08,Withdrawn,5,Group,109,-0.664735,0,4,Medium,Neglected
2644,57,SH07,Lighthouse Safehouse 7,2026-02-17,Happy,8,Individual,75,-0.361430,0,1,Low,Abandoned
1423,26,SH03,Lighthouse Safehouse 3,2024-10-25,Withdrawn,5,Individual,58,-0.181500,0,5,Low,Abandoned
1155,22,SH06,Lighthouse Safehouse 6,2024-11-20,Happy,8,Group,113,-0.101594,0,1,Medium,Abandoned
2818,60,SH07,Lighthouse Safehouse 7,2025-11-15,Withdrawn,5,Individual,75,0.126527,0,3,Medium,Abandoned
2520,53,SH06,Lighthouse Safehouse 6,2026-02-13,Hopeful,7,Group,67,0.170410,1,3,Low,Surrendered
1612,30,SH05,Lighthouse Safehouse 5,2025-08-09,Hopeful,7,Individual,90,0.183901,1,5,Critical,Abandoned
